# Lesson 40: Agentic RAG — letting an agent decide how to look things up 🕵️📚
> 🎓 **Advanced track** — best for Years 11–13. Students should be comfortable with Python functions, lists and dictionaries.

**Learning goals**
1. Explain how **agentic RAG** differs from plain RAG
2. **Route** questions to the right knowledge base
3. Break comparison questions into **sub-questions**
4. Build a **LlamaIndex agent** that chooses between several retrieval tools and a calculator, remembers the conversation, and shows its steps

**⏱️ Timing:** plain vs agentic 10' | router 10' | sub-questions 10' | the agent 25' | review 5'

> ⚙️ **GPU required:** Runtime → Change runtime type → **T4 GPU**. Agents need a smarter model, so this lesson uses `qwen2.5:7b` (about 5 GB to download). Change it to `qwen2.5:3b` for a faster but less reliable run.

## 🧭 Plain RAG vs agentic RAG
**Plain RAG** (Lesson 32) is a fixed recipe: *every* question → retrieve top-k from *one* store → answer. It doesn't think about **whether**, **where** or **how many times** to search.

**Agentic RAG** puts an **agent** (Lesson 35) in charge of retrieval. It can:
| Decision | Example |
|---|---|
| **Whether** to retrieve at all | "Hi!" doesn't need a search |
| **Where** to search | School rules → handbook; biology → science notes |
| **How** to split the question | "Compare photosynthesis and respiration" → two searches |
| **When to search again** | Results not good enough → rephrase and retry (Lesson 34) |
| **Which other tools** to use | A calculator for "how much does printing 37 pages cost?" |

> Analogy: plain RAG is a vending machine (press button → get snack). Agentic RAG is a **librarian** who asks which section you need, checks several shelves and fetches a calculator if needed.

In [ ]:
# 🦙 Start a free LLM server *inside* Colab (Ollama). No API key or account needed.
# ⏳ First run: about 2–3 minutes (installs Ollama and downloads the model).
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
import subprocess, time
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

MODEL = "qwen2.5:7b"     # smarter at using tools. "qwen2.5:3b" downloads faster but makes more mistakes
!ollama pull {MODEL} > /dev/null 2>&1
print("✅ Ollama is running with", MODEL)

In [ ]:
!pip -q install llama-index-core llama-index-llms-ollama llama-index-embeddings-huggingface
import os, warnings; warnings.filterwarnings("ignore")
from llama_index.core import Settings, SimpleDirectoryReader, VectorStoreIndex
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

Settings.llm = Ollama(model=MODEL, request_timeout=300, temperature=0)
Settings.embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
os.makedirs("handbook", exist_ok=True); os.makedirs("science_notes", exist_ok=True)
files = {
 "handbook/attendance.md": "Students must arrive by 8:40am. Late students sign in at the main office. Parents report sickness on the attendance line before 9am.",
 "handbook/devices.md": "Phones must be in lockers during class. Phones may be used at interval and lunch outdoors. Confiscated phones are collected at 3:15pm.",
 "handbook/library.md": "The library is open 8:00am-4:30pm on weekdays. Students may borrow 5 books for three weeks. Printing costs 10 cents a page.",
 "handbook/assessment.md": "Extensions must be requested 2 school days before the due date. AI tools may be used only when the task sheet allows it, and AI use must be acknowledged.",
 "science_notes/photosynthesis.md": "Photosynthesis: plants use light energy, water and carbon dioxide to make glucose and oxygen. It happens in chloroplasts, which contain chlorophyll.",
 "science_notes/respiration.md": "Respiration: cells break down glucose with oxygen to release energy, producing carbon dioxide and water. It happens in mitochondria, in plants and animals.",
 "science_notes/forces.md": "Forces: a force is a push or pull measured in newtons (N). Weight = mass x 9.8 N/kg on Earth. Friction opposes motion.",
 "science_notes/acids.md": "Acids have a pH below 7, bases above 7. Neutralisation: acid + base -> salt + water. Universal indicator turns red in strong acid and purple in strong base.",
}
for path, text in files.items():
    open(path, "w").write(text)
print("Wrote", len(files), "files")

In [ ]:
handbook_index = VectorStoreIndex.from_documents(SimpleDirectoryReader("handbook").load_data())
science_index = VectorStoreIndex.from_documents(SimpleDirectoryReader("science_notes").load_data())
handbook_qe = handbook_index.as_query_engine(similarity_top_k=2)
science_qe = science_index.as_query_engine(similarity_top_k=2)

# Plain RAG over ONE store can't find facts that live in the other store:
r = handbook_qe.query("Where does respiration happen in a cell?")
print("Plain RAG (handbook only) on a science question:\n ", r)
print("Sources it used:", [n.metadata["file_name"] for n in r.source_nodes])
# 👀 The sources are handbook files! If the answer sounds right, the model used its own memory, not your notes —
#    so there's no source to check it against. That's the risk agentic RAG fixes by searching the right store.

## Part 1: Routing — choose the right knowledge base 🔀
Each store becomes a **tool** with a **description**. A router asks the LLM which tool fits the question.

In [ ]:
from llama_index.core.tools import QueryEngineTool
from llama_index.core.query_engine import RouterQueryEngine
from llama_index.core.selectors import LLMSingleSelector

handbook_tool = QueryEngineTool.from_defaults(handbook_qe, name="school_handbook",
    description="School rules: attendance, lateness, phones, library opening hours and borrowing, printing costs, assessment and AI use.")
science_tool = QueryEngineTool.from_defaults(science_qe, name="science_notes",
    description="Year 10 science notes: photosynthesis, respiration, forces and weight, acids, bases and pH.")

router = RouterQueryEngine(selector=LLMSingleSelector.from_defaults(), query_engine_tools=[handbook_tool, science_tool], verbose=True)
for q in ["How many books can I borrow?", "What is the weight of a 50 kg person on Earth?"]:
    print(f"\n❓ {q}\n🤖 {router.query(q)}")

### 🧩 Exercise 1
Make the tool **descriptions** vague (e.g. `"some documents"`). Does routing still work? In agentic RAG, **descriptions are the instructions**.

## Part 2: Sub-questions — split, search, combine 🧩
"Compare X and Y" needs facts from two places. The **sub-question engine** breaks it apart, answers each piece with the right tool, then combines the answers.

In [ ]:
from llama_index.core.query_engine import SubQuestionQueryEngine
from llama_index.core.question_gen import LLMQuestionGenerator

subq = SubQuestionQueryEngine.from_defaults(query_engine_tools=[handbook_tool, science_tool],
                                            question_gen=LLMQuestionGenerator.from_defaults(), verbose=True)
print(subq.query("Compare photosynthesis and respiration: what goes in, what comes out, and where does each happen?"))

## Part 3: The agentic RAG assistant 🤖
Now a full **agent**: it has both knowledge bases **and** a calculator, decides which to use (maybe several, maybe none), and keeps **memory** of the conversation with a `Context`. We'll print each tool call so you can watch it think.

In [ ]:
from llama_index.core.tools import FunctionTool
from llama_index.core.agent.workflow import FunctionAgent, ToolCall, ToolCallResult
from llama_index.core.workflow import Context

def calculate(expression: str) -> str:
    """Evaluate a maths expression with numbers and + - * / ( ). Example: '37 * 0.10'."""
    import re
    if not re.fullmatch(r"[\d\s+\-*/().]+", expression): return "error: maths only"
    return str(round(eval(expression), 4))

agent = FunctionAgent(
    tools=[handbook_tool, science_tool, FunctionTool.from_defaults(calculate)],
    llm=Settings.llm,
    system_prompt=("You are a school study assistant. Use school_handbook for school rules and science_notes for science facts. "
                   "Use calculate for any arithmetic. Answer only from tool results; if they don't contain the answer, say so. "
                   "Don't use tools for greetings or small talk."))
ctx = Context(agent)       # conversation memory

async def chat(message):
    print(f"🧑 {message}")
    handler = agent.run(message, ctx=ctx)
    async for event in handler.stream_events():
        if isinstance(event, ToolCall):
            print(f"   🔧 {event.tool_name}({event.tool_kwargs})")
        elif isinstance(event, ToolCallResult):
            print(f"   📄 → {str(event.tool_output)[:120]}")
    print(f"🤖 {await handler}\n")

await chat("Hi there!")
await chat("How much would it cost to print 37 pages at the library?")
await chat("What is the weight of a 65 kg student, and is it measured in kg or N?")

In [ ]:
await chat("Which of the two processes we studied releases energy, and where in the cell?")
await chat("If I need an extension for my science report due Friday, by which day must I ask?")
await chat("What did I ask you about printing earlier?")

### 🧩 Exercise 2: Plain vs agentic
1. For each question, write down which tools were called. Were any calls **unnecessary** or **missing**?
2. Ask something that's in **neither** store (e.g. "What's the canteen menu?"). Does the agent admit it doesn't know?
3. Count the LLM calls. Agentic RAG is smarter but **slower and more expensive** than plain RAG. When is it worth it?

### 🚀 Challenge
- Add a third knowledge base (e.g. a `maths_notes` folder) and a tool for it. Did you need to change the agent's code, or just its tool list?
- Add a **self-check** step: after answering, ask the LLM "Is every claim supported by the tool results?" and retry if not (Lesson 34).

## 📌 Summary
- **Agentic RAG** = an agent decides **whether, where, how and how often** to retrieve, and can use other tools
- **Routing** sends a question to the right store; **sub-questions** split complex questions
- **Tool descriptions** guide the agent; **Context** gives it memory
- Smarter, but costs more calls — measure before you choose